<a href="https://colab.research.google.com/github/himasree-d/NLP-Project/blob/main/subtitle_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

STEP 1 — Setup + Upload File + Convert File

In [ ]:
!pip install -U faster-whisper librosa noisereduce soundfile ffmpeg-python moviepy jiwer pysrt


AUDIO FORMATS SUPPORTED
The code supports ALL common audio formats, including:
.mp3  .wav  .m4a  .aac  .ogg  .flac  .wma

VIDEO FORMATS SUPPORTED
These are supported for automatic audio extraction:
.mp4 .mkv  .avi  .mov  .flv  .wmv  .3gp  .mpeg  .webm

In [ ]:
from google.colab import files
uploaded = files.upload()

for fname in uploaded.keys():
    file_path = "/content/" + fname
    print("Uploaded:", file_path)


Saving 2025-02-03 16_57_41_1738598261.49391.mp3 to 2025-02-03 16_57_41_1738598261.49391.mp3
Uploaded: /content/2025-02-03 16_57_41_1738598261.49391.mp3


In [ ]:
import ffmpeg

def convert_to_wav(path):
    out = "/content/converted.wav"

    # Extract audio from ANY video or audio file using FFmpeg only
    (
        ffmpeg
        .input(path)
        .output(out, ac=1, ar=16000)  # mono, 16kHz
        .overwrite_output()
        .run(quiet=True)
    )

    print("Converted to 16kHz WAV:", out)
    return out

audio_path = convert_to_wav(file_path)


Converted to 16kHz WAV: /content/converted.wav


STEP 2 — AUDIO PREPROCESSING

Load Audio + Noise Reduction

In [ ]:
import librosa
import noisereduce as nr

y, sr = librosa.load(audio_path, sr=16000, mono=True)

print("Original audio length:", round(len(y)/sr, 2), "seconds")

# Strong noise reduction for multilingual speech
y_nr = nr.reduce_noise(y=y, sr=sr, prop_decrease=0.9)

print("Noise reduction done.")


Original audio length: 433.58 seconds
Noise reduction done.


Silence Removal (Stable VAD)

In [ ]:
import numpy as np

intervals = librosa.effects.split(y_nr, top_db=32)

if len(intervals) == 0:
    cleaned = y_nr
    print(" No speech intervals detected — using noise-reduced audio.")
else:
    cleaned = np.concatenate([y_nr[s:e] for s,e in intervals])
    print("Speech segments detected:", len(intervals))


Speech segments detected: 518


Normalize Audio + Save Clean File

In [ ]:
import soundfile as sf

if np.max(np.abs(cleaned)) > 0:
    cleaned = cleaned / np.max(np.abs(cleaned))

cleaned_path = "/content/cleaned.wav"
sf.write(cleaned_path, cleaned, sr)

print("Cleaned audio saved:", cleaned_path)
print("Cleaned audio length:", round(len(cleaned)/sr, 2), "seconds")


Cleaned audio saved: /content/cleaned.wav
Cleaned audio length: 287.2 seconds


Create Chunks

In [ ]:
import os, math

def create_chunks(cleaned_audio, sr, chunk_sec=30):
    duration = librosa.get_duration(y=cleaned_audio, sr=sr)
    num_chunks = math.ceil(duration / chunk_sec)

    os.makedirs("/content/chunks", exist_ok=True)
    chunk_list = []

    for i in range(num_chunks):
        start = int(i * chunk_sec * sr)
        end   = int(min((i+1)*chunk_sec*sr, len(cleaned_audio)))

        part = cleaned_audio[start:end]
        path = f"/content/chunks/chunk_{i:03d}.wav"
        sf.write(path, part, sr)

        chunk_list.append((path, i * chunk_sec))

    print("Chunks created:", len(chunk_list))
    return chunk_list

chunks = create_chunks(cleaned, sr)


Chunks created: 10


STEP 3 — Full Transcription Pipeline + SRT Output

Load Whisper Large-V3 (Best Multilingual Accuracy)

In [ ]:
from faster_whisper import WhisperModel

print("Loading Whisper Large-V3 model...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

print("Model loaded.")


Loading Whisper Large-V3 model...
Model loaded.


Transcribe All Chunks + PRINT PROBABILITY

In [ ]:
all_segments = []

for path, offset in chunks:
    segments, info = model.transcribe(
        path,
        task="transcribe",
        vad_filter=False
    )

    print("\nChunk:", path)
    print("Detected Language:", info.language)
    print("Probability:", info.language_probability)

    for s in segments:
        all_segments.append({
            "start": s.start + offset,
            "end":   s.end + offset,
            "text":  s.text.strip()
        })

print("\nTotal segments:", len(all_segments))



Chunk: /content/chunks/chunk_000.wav
Detected Language: fr
Probability: 0.98486328125

Chunk: /content/chunks/chunk_001.wav
Detected Language: fr
Probability: 0.9814453125

Chunk: /content/chunks/chunk_002.wav
Detected Language: fr
Probability: 0.9873046875

Chunk: /content/chunks/chunk_003.wav
Detected Language: fr
Probability: 0.9814453125

Chunk: /content/chunks/chunk_004.wav
Detected Language: fr
Probability: 0.98583984375

Chunk: /content/chunks/chunk_005.wav
Detected Language: fr
Probability: 0.994140625

Chunk: /content/chunks/chunk_006.wav
Detected Language: fr
Probability: 0.9931640625

Chunk: /content/chunks/chunk_007.wav
Detected Language: fr
Probability: 0.98876953125

Chunk: /content/chunks/chunk_008.wav
Detected Language: fr
Probability: 0.98876953125

Chunk: /content/chunks/chunk_009.wav
Detected Language: fr
Probability: 0.9892578125

Total segments: 177


Build inital SRT

In [ ]:
def to_srt(sec):
    ms = int((sec - int(sec)) * 1000)
    sec = int(sec)
    return f"{sec//3600:02}:{(sec//60)%60:02}:{sec%60:02},{ms:03}"

initial_segments = sorted(all_segments, key=lambda x: x["start"])

srt_lines = []
idx = 1

for seg in initial_segments:
    if not seg["text"]:
        continue
    srt_lines.append(str(idx))
    srt_lines.append(f"{to_srt(seg['start'])} --> {to_srt(seg['end'])}")
    srt_lines.append(seg["text"])
    srt_lines.append("")
    idx += 1

initial_srt_path = "/content/initial_raw_output.srt"

with open(initial_srt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

print("INITIAL SRT created:", initial_srt_path)


INITIAL SRT created: /content/initial_raw_output.srt


In [ ]:
from google.colab import files
files.download(initial_srt_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

STEP 4

Safe Bandpass Filter

In [ ]:
import scipy.signal as sps
import numpy as np

def bandpass_filter(audio, sr, low_hz=70, high_hz=8000):
    nyq = sr/2
    if high_hz >= nyq:
        high_hz = nyq - 100

    low = low_hz / nyq
    high = high_hz / nyq

    if not (0 < low < high < 1):
        return audio

    b, a = sps.butter(4, [low, high], btype='band')
    return sps.filtfilt(b, a, audio)

filtered_audio = bandpass_filter(cleaned, sr)


Strong Noise Reduction

In [ ]:
filtered_nr = nr.reduce_noise(y=filtered_audio, sr=sr, prop_decrease=1.0)


Text Cleaner (Language-agnostic)

In [ ]:
import re

def clean_text(t):
    t = t.strip()
    t = re.sub(r"[“”\"\'\(\)\[\]<>]", "", t)
    t = re.sub(r"\.{2,}", ".", t)
    t = re.sub(r",{2,}", ",", t)

    fillers = ["uh", "um", "hmm", "erm"]
    for f in fillers:
        t = re.sub(r"\b"+f+r"\b", "", t, flags=re.IGNORECASE)

    return re.sub(r"\s+", " ", t).strip()


Indic Script Normalization (Telugu/Hindi/Tamil/Kannada/Malayalam)

In [ ]:
def fix_indic_script(t):
    # remove zero-width characters
    t = re.sub(r"[\u200B-\u200D\uFEFF]", "", t)

    # fix Hindi/Telugu/Tamil virama spacing
    t = re.sub(r"\u094D\s+", "\u094D", t)  # Hindi
    t = re.sub(r"\u0C4D\s+", "\u0C4D", t)  # Telugu
    t = re.sub(r"\u0BCD\s+", "\u0BCD", t)  # Tamil

    return t


Apply Cleaning to ALL Segments

In [ ]:
for seg in all_segments:
    text = seg["text"]
    text = clean_text(text)
    text = fix_indic_script(text)
    seg["text"] = text


In [ ]:
cleaned_segments = sorted(all_segments, key=lambda x: x["start"])

srt_lines = []
idx = 1

for seg in cleaned_segments:
    if not seg["text"]:
        continue

    srt_lines.append(str(idx))
    srt_lines.append(f"{to_srt(seg['start'])} --> {to_srt(seg['end'])}")
    srt_lines.append(seg["text"])
    srt_lines.append("")
    idx += 1

final_srt_path = "/content/final_clean_output.srt"

with open(final_srt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

print("FINAL CLEAN SRT created:", final_srt_path)

FINAL CLEAN SRT created: /content/final_clean_output.srt


In [ ]:
from google.colab import files
files.download(final_srt_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>